# Financial Statements Data Fetcher

**Purpose:** Fetch quarterly financial statements (income, balance sheet, cash flow) for ~3,000 US stocks from Financial Modeling Prep API.

**Coverage:** 2016-2026 (40 quarters = 10 years)

**Outputs:**
- `data/financial_statements_final/income_statements.parquet`
- `data/financial_statements_final/balance_sheets.parquet`
- `data/financial_statements_final/cash_flows.parquet`

**Features:**
- Progress reports every 50 tickers
- Resume support (caching per ticker)
- Rate limiting (240 calls/minute with 1 worker = 80 tickers/min)
- Retry logic for rate limit errors (up to 3 attempts)
- ~35 minute runtime for full dataset

## Cell 1: Imports & Configuration

In [10]:
import pandas as pd
import numpy as np
import requests
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Configuration
API_KEY = "UdRiHSXViYG3qI0ChXrlhVRvqOPXRAB5"
CACHE_DIR = "data/financial_statements_cache"
OUTPUT_DIR = "data/financial_statements_final"
PERIOD = "quarter"
LIMIT = 40  # 10 years of quarterly data (2016-2026)
MAX_WORKERS = 1  # Single worker to avoid parallel rate limit issues
RATE_LIMIT_DELAY = 0.25  # 1 worker × 4 calls/sec = 240 API calls/min = 80 tickers/min
MAX_RETRIES = 3  # Retry on rate limit errors

# Create directories
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Configuration loaded")
print(f"   API Key: {API_KEY[:20]}...")
print(f"   Cache: {CACHE_DIR}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Period: {PERIOD}, Limit: {LIMIT}")
print(f"   API calls/min: {1/RATE_LIMIT_DELAY*60:.0f} (= ~{1/RATE_LIMIT_DELAY*60/3:.0f} tickers/min)")
print(f"   Workers: {MAX_WORKERS}")
print(f"   Max retries: {MAX_RETRIES}")

✅ Configuration loaded
   API Key: UdRiHSXViYG3qI0ChXrl...
   Cache: data/financial_statements_cache
   Output: data/financial_statements_final
   Period: quarter, Limit: 40
   API calls/min: 240 (= ~80 tickers/min)
   Workers: 1
   Max retries: 3


## Cell 2: API Fetcher Functions

In [11]:
def fetch_statement(ticker: str, statement_type: str, api_key: str,
                   period: str = "quarter", limit: int = 40, max_retries: int = 3) -> pd.DataFrame:
    """
    Fetch single financial statement for one ticker with retry logic.
    
    Args:
        ticker: Stock symbol
        statement_type: 'income-statement', 'balance-sheet-statement', 'cash-flow-statement'
        api_key: FMP API key
        period: 'quarter' or 'annual'
        limit: Number of periods to fetch (40 = 10 years quarterly)
        max_retries: Number of retries on rate limit errors
    
    Returns:
        DataFrame with all columns from API response
    """
    # Check cache first
    cache_file = f"{CACHE_DIR}/{ticker}_{statement_type}.parquet"
    if os.path.exists(cache_file):
        return pd.read_parquet(cache_file)
    
    # Fetch from API with retry logic
    url = f"https://financialmodelingprep.com/stable/{statement_type}"
    params = {
        "symbol": ticker,
        "period": period,
        "limit": limit,
        "apikey": api_key
    }
    
    for attempt in range(max_retries):
        try:
            time.sleep(RATE_LIMIT_DELAY)  # Rate limiting (240 calls/min)
            response = requests.get(url, params=params, timeout=30)
            
            # Handle rate limit errors (429) - retry with backoff
            if response.status_code == 429:
                wait_time = (2 ** attempt) * 5  # Exponential backoff: 5s, 10s, 20s
                print(f"⚠️  Rate limit hit for {ticker} {statement_type}, waiting {wait_time}s...")
                time.sleep(wait_time)
                continue  # Retry
            
            # Other HTTP errors - don't retry
            if response.status_code != 200:
                return pd.DataFrame()
            
            data = response.json()
            
            # Error checking
            if isinstance(data, dict) and 'Error Message' in data:
                return pd.DataFrame()
            
            if not data:
                return pd.DataFrame()
            
            # Convert to DataFrame (keep ALL columns)
            df = pd.DataFrame(data)
            df['ticker'] = ticker
            
            # Cache result
            df.to_parquet(cache_file, index=False)
            
            return df
        
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) * 2
                print(f"⚠️  Error fetching {ticker} {statement_type} (attempt {attempt+1}): {e}, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed {ticker} {statement_type} after {max_retries} attempts: {e}")
                return pd.DataFrame()
    
    return pd.DataFrame()


def fetch_all_statements(ticker: str, api_key: str) -> dict:
    """
    Fetch all three financial statements for one ticker.
    
    Returns:
        Dict with keys: 'income', 'balance', 'cashflow', 'success' (bool)
    """
    income = fetch_statement(ticker, 'income-statement', api_key)
    balance = fetch_statement(ticker, 'balance-sheet-statement', api_key)
    cashflow = fetch_statement(ticker, 'cash-flow-statement', api_key)
    
    # ALL 3 must succeed
    success = (not income.empty) and (not balance.empty) and (not cashflow.empty)
    
    return {
        'income': income,
        'balance': balance,
        'cashflow': cashflow,
        'success': success
    }


def load_processed_tickers(cache_dir: str) -> set:
    """Load set of tickers that have ALL 3 statements cached."""
    if not os.path.exists(cache_dir):
        return set()
    
    processed = set()
    
    # Only count as processed if all 3 files exist
    for file in os.listdir(cache_dir):
        if file.endswith('_income-statement.parquet'):
            ticker = file.replace('_income-statement.parquet', '')
            
            # Check all 3 exist
            income_file = f"{cache_dir}/{ticker}_income-statement.parquet"
            balance_file = f"{cache_dir}/{ticker}_balance-sheet-statement.parquet"
            cashflow_file = f"{cache_dir}/{ticker}_cash-flow-statement.parquet"
            
            if (os.path.exists(income_file) and 
                os.path.exists(balance_file) and 
                os.path.exists(cashflow_file)):
                processed.add(ticker)
    
    return processed


def filter_remaining_tickers(all_tickers: list, cache_dir: str) -> list:
    """Return only tickers that don't have all 3 statements cached yet."""
    processed = load_processed_tickers(cache_dir)
    remaining = [t for t in all_tickers if t not in processed]
    
    return remaining


print("✅ API fetcher functions defined (with retry logic + requires all 3 statements)")

✅ API fetcher functions defined (with retry logic + requires all 3 statements)


## Cell 3: Parallel Fetcher with Progress Tracking

In [12]:
def fetch_all_tickers(tickers: list, api_key: str, max_workers: int = 1) -> tuple:
    """
    Fetch financial statements for all tickers in parallel.
    
    Args:
        tickers: List of ticker symbols
        api_key: FMP API key
        max_workers: Number of parallel threads
    
    Returns:
        Tuple: (all_data dict, failed_tickers list)
    """
    all_data = {}
    failed_tickers = []
    partial_failures = {}  # Track which statements failed
    completed_count = 0
    
    def fetch_ticker(ticker):
        statements = fetch_all_statements(ticker, api_key)
        return ticker, statements
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_ticker, ticker): ticker
                  for ticker in tickers}
        
        for future in tqdm(as_completed(futures), total=len(futures),
                          desc="Fetching statements"):
            ticker, statements = future.result()
            completed_count += 1
            
            # ALL 3 statements must succeed
            if statements['success']:
                all_data[ticker] = statements
            else:
                # Track which statements failed
                missing = []
                if statements['income'].empty:
                    missing.append('income')
                if statements['balance'].empty:
                    missing.append('balance')
                if statements['cashflow'].empty:
                    missing.append('cashflow')
                
                failed_tickers.append(ticker)
                partial_failures[ticker] = missing
            
            # Progress report every 50 tickers
            if completed_count % 50 == 0:
                success_count = len(all_data)
                fail_count = len(failed_tickers)
                success_rate = (success_count / completed_count) * 100
                
                print(f"\n{'='*80}")
                print(f"PROGRESS REPORT - {completed_count}/{len(tickers)} tickers completed")
                print(f"{'='*80}")
                print(f"  ✅ Successful: {success_count} ({success_rate:.1f}%) - ALL 3 statements")
                print(f"  ❌ Failed: {fail_count}")
                if failed_tickers:
                    print(f"  Recent failures: {failed_tickers[-5:]}")
                    # Show what failed for last failure
                    last_fail = failed_tickers[-1]
                    if last_fail in partial_failures:
                        print(f"    {last_fail} missing: {partial_failures[last_fail]}")
                print(f"{'='*80}\n")
    
    # Final summary of partial failures
    if partial_failures:
        print(f"\n{'='*80}")
        print(f"PARTIAL FAILURE SUMMARY")
        print(f"{'='*80}")
        income_fails = sum(1 for v in partial_failures.values() if 'income' in v)
        balance_fails = sum(1 for v in partial_failures.values() if 'balance' in v)
        cashflow_fails = sum(1 for v in partial_failures.values() if 'cashflow' in v)
        print(f"  Income statement failures: {income_fails}")
        print(f"  Balance sheet failures: {balance_fails}")
        print(f"  Cash flow failures: {cashflow_fails}")
        print(f"{'='*80}\n")
    
    return all_data, failed_tickers


print("✅ Parallel fetcher function defined (requires ALL 3 statements)")

✅ Parallel fetcher function defined (requires ALL 3 statements)


## Cell 4: Load Cached Data Function

In [13]:
def load_all_cached_statements(cache_dir: str) -> dict:
    """
    Load all cached statements from parquet files.
    
    Returns:
        Dict mapping ticker -> {income, balance, cashflow DataFrames}
    """
    all_data = {}
    
    # Get unique tickers from cached income statements
    tickers = set()
    for file in os.listdir(cache_dir):
        if file.endswith('_income-statement.parquet'):
            ticker = file.replace('_income-statement.parquet', '')
            tickers.add(ticker)
    
    # Load all three statements for each ticker
    for ticker in tqdm(tickers, desc="Loading cached data"):
        try:
            income_file = f"{cache_dir}/{ticker}_income-statement.parquet"
            balance_file = f"{cache_dir}/{ticker}_balance-sheet-statement.parquet"
            cashflow_file = f"{cache_dir}/{ticker}_cash-flow-statement.parquet"
            
            statements = {}
            
            if os.path.exists(income_file):
                statements['income'] = pd.read_parquet(income_file)
            else:
                statements['income'] = pd.DataFrame()
            
            if os.path.exists(balance_file):
                statements['balance'] = pd.read_parquet(balance_file)
            else:
                statements['balance'] = pd.DataFrame()
            
            if os.path.exists(cashflow_file):
                statements['cashflow'] = pd.read_parquet(cashflow_file)
            else:
                statements['cashflow'] = pd.DataFrame()
            
            # Only include if at least income statement exists
            if not statements['income'].empty:
                all_data[ticker] = statements
        
        except Exception as e:
            print(f"Error loading {ticker}: {e}")
    
    return all_data


print("✅ Cache loader function defined")

✅ Cache loader function defined


## Cell 5: Save Combined Statements Function

In [14]:
def save_combined_statements(all_data: dict, output_dir: str):
    """
    Combine all tickers and save to 3 separate parquet files.
    
    Args:
        all_data: Dict mapping ticker -> {income, balance, cashflow}
        output_dir: Output directory path
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Combine each statement type
    income_dfs = []
    balance_dfs = []
    cashflow_dfs = []
    
    for ticker, statements in all_data.items():
        if not statements['income'].empty:
            income_dfs.append(statements['income'])
        if not statements['balance'].empty:
            balance_dfs.append(statements['balance'])
        if not statements['cashflow'].empty:
            cashflow_dfs.append(statements['cashflow'])
    
    # Save income statements
    if income_dfs:
        income_combined = pd.concat(income_dfs, ignore_index=True)
        income_combined['date'] = pd.to_datetime(income_combined['date'])
        income_combined = income_combined.sort_values(['symbol', 'date'],
                                                      ascending=[True, False])
        income_combined.to_parquet(f"{output_dir}/income_statements.parquet",
                                   index=False)
        print(f"✅ Saved income_statements.parquet: {income_combined.shape}")
        print(f"   Tickers: {income_combined['symbol'].nunique()}")
        print(f"   Date range: {income_combined['date'].min()} to {income_combined['date'].max()}")
    
    # Save balance sheets
    if balance_dfs:
        balance_combined = pd.concat(balance_dfs, ignore_index=True)
        balance_combined['date'] = pd.to_datetime(balance_combined['date'])
        balance_combined = balance_combined.sort_values(['symbol', 'date'],
                                                        ascending=[True, False])
        balance_combined.to_parquet(f"{output_dir}/balance_sheets.parquet",
                                    index=False)
        print(f"✅ Saved balance_sheets.parquet: {balance_combined.shape}")
        print(f"   Tickers: {balance_combined['symbol'].nunique()}")
        print(f"   Date range: {balance_combined['date'].min()} to {balance_combined['date'].max()}")
    
    # Save cash flows
    if cashflow_dfs:
        cashflow_combined = pd.concat(cashflow_dfs, ignore_index=True)
        cashflow_combined['date'] = pd.to_datetime(cashflow_combined['date'])
        cashflow_combined = cashflow_combined.sort_values(['symbol', 'date'],
                                                          ascending=[True, False])
        cashflow_combined.to_parquet(f"{output_dir}/cash_flows.parquet",
                                     index=False)
        print(f"✅ Saved cash_flows.parquet: {cashflow_combined.shape}")
        print(f"   Tickers: {cashflow_combined['symbol'].nunique()}")
        print(f"   Date range: {cashflow_combined['date'].min()} to {cashflow_combined['date'].max()}")


print("✅ Save function defined")

✅ Save function defined


## Cell 6: Check Resume Status

In [15]:
# Load ticker universe
stocks_df = pd.read_csv('us_stocks_500m.csv')
all_tickers = stocks_df['ticker'].tolist()

# Filter to remaining tickers (resume support)
remaining_tickers = filter_remaining_tickers(all_tickers, CACHE_DIR)

print(f"\n{'='*80}")
print(f"TICKER UNIVERSE")
print(f"{'='*80}")
print(f"Total tickers: {len(all_tickers)}")
print(f"Already cached: {len(all_tickers) - len(remaining_tickers)}")
print(f"Remaining to fetch: {len(remaining_tickers)}")
print(f"API calls needed: {len(remaining_tickers) * 3}")
print(f"Estimated time: {len(remaining_tickers) * 3 / 300:.1f} minutes")
print(f"{'='*80}\n")


TICKER UNIVERSE
Total tickers: 3098
Already cached: 3081
Remaining to fetch: 17
API calls needed: 51
Estimated time: 0.2 minutes



## Cell 7: Fetch Statements

In [16]:
if len(remaining_tickers) == 0:
    print("✅ All tickers already cached! Skipping to combine step...")
else:
    print(f"🚀 Starting fetch for {len(remaining_tickers)} tickers...\n")
    
    # Fetch data with progress reporting
    all_data, failed = fetch_all_tickers(remaining_tickers, API_KEY, MAX_WORKERS)
    
    print(f"\n{'='*80}")
    print(f"FETCH COMPLETE")
    print(f"{'='*80}")
    print(f"  ✅ Successful: {len(all_data)}")
    print(f"  ❌ Failed: {len(failed)}")
    if failed:
        print(f"  Failed tickers: {failed[:20]}")
        if len(failed) > 20:
            print(f"  ... and {len(failed) - 20} more")
    print(f"{'='*80}\n")

🚀 Starting fetch for 17 tickers...



Fetching statements: 100%|██████████| 17/17 [00:11<00:00,  1.42it/s]



PARTIAL FAILURE SUMMARY
  Income statement failures: 10
  Balance sheet failures: 3
  Cash flow failures: 16


FETCH COMPLETE
  ✅ Successful: 1
  ❌ Failed: 16
  Failed tickers: ['VSNTV', 'BRBI', 'ELVR', 'NAVN', 'NP', 'WRD', 'HBNB', 'MAAS', 'DGNX', 'FSCO', 'BTQ', 'BTX', 'THH', 'WBI', 'DMIIU', 'BCSS']



## Cell 8: Load All Cached Data & Combine

In [17]:
# Load ALL cached data (including newly fetched + previously cached)
all_cached_data = load_all_cached_statements(CACHE_DIR)

print(f"\n✅ Loaded {len(all_cached_data)} tickers from cache")

# Combine and save to 3 final parquet files
print(f"\nCombining and saving to parquet files...\n")
save_combined_statements(all_cached_data, OUTPUT_DIR)

print(f"\n✅ Complete! Data saved to {OUTPUT_DIR}/")

Loading cached data: 100%|██████████| 3088/3088 [00:30<00:00, 99.72it/s] 



✅ Loaded 3088 tickers from cache

Combining and saving to parquet files...

✅ Saved income_statements.parquet: (109961, 40)
   Tickers: 3088
   Date range: 1999-01-31 00:00:00 to 2026-01-26 00:00:00
✅ Saved balance_sheets.parquet: (108319, 62)
   Tickers: 3088
   Date range: 1983-06-30 00:00:00 to 2026-01-25 00:00:00
✅ Saved cash_flows.parquet: (109348, 48)
   Tickers: 3082
   Date range: 2001-03-31 00:00:00 to 2026-01-24 00:00:00

✅ Complete! Data saved to data/financial_statements_final/


## Cell 9: Verification & Statistics

In [18]:
# Load combined files
income = pd.read_parquet(f'{OUTPUT_DIR}/income_statements.parquet')
balance = pd.read_parquet(f'{OUTPUT_DIR}/balance_sheets.parquet')
cashflow = pd.read_parquet(f'{OUTPUT_DIR}/cash_flows.parquet')

print(f"{'='*80}")
print(f"FINAL DATA VERIFICATION")
print(f"{'='*80}\n")

print(f"Income statements: {income.shape}")
print(f"Balance sheets: {balance.shape}")
print(f"Cash flows: {cashflow.shape}")

# Check date range and tickers
print(f"\nIncome Statements:")
print(f"  Date range: {income['date'].min()} to {income['date'].max()}")
print(f"  Unique tickers: {income['symbol'].nunique()}")
print(f"  Columns: {len(income.columns)}")
print(f"  First 20 columns: {list(income.columns[:20])}")

print(f"\nBalance Sheets:")
print(f"  Date range: {balance['date'].min()} to {balance['date'].max()}")
print(f"  Unique tickers: {balance['symbol'].nunique()}")
print(f"  Columns: {len(balance.columns)}")

print(f"\nCash Flows:")
print(f"  Date range: {cashflow['date'].min()} to {cashflow['date'].max()}")
print(f"  Unique tickers: {cashflow['symbol'].nunique()}")
print(f"  Columns: {len(cashflow.columns)}")

# Check for AAPL example (40 quarters expected)
aapl_income = income[income['symbol'] == 'AAPL'].sort_values('date', ascending=False)
print(f"\n{'='*80}")
print(f"AAPL EXAMPLE (Expected ~40 quarters)")
print(f"{'='*80}")
print(f"\nAAPL quarters in income statement: {len(aapl_income)}")
print(f"\nAAPL recent quarters:")
display(aapl_income[['date', 'symbol', 'period', 'fiscalYear', 'revenue', 'netIncome']].head(10))

# Verify key columns exist
print(f"\n{'='*80}")
print(f"COLUMN VALIDATION")
print(f"{'='*80}")
required_cols = ['date', 'symbol', 'period', 'fiscalYear']
for df_name, df in [('Income', income), ('Balance', balance), ('Cashflow', cashflow)]:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"⚠️  {df_name} missing columns: {missing}")
    else:
        print(f"✅ {df_name} has all required columns")

print(f"\n{'='*80}")
print(f"✅ VERIFICATION COMPLETE")
print(f"{'='*80}")

FINAL DATA VERIFICATION

Income statements: (109961, 40)
Balance sheets: (108319, 62)
Cash flows: (109348, 48)

Income Statements:
  Date range: 1999-01-31 00:00:00 to 2026-01-26 00:00:00
  Unique tickers: 3088
  Columns: 40
  First 20 columns: ['date', 'symbol', 'reportedCurrency', 'cik', 'filingDate', 'acceptedDate', 'fiscalYear', 'period', 'revenue', 'costOfRevenue', 'grossProfit', 'researchAndDevelopmentExpenses', 'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses', 'sellingGeneralAndAdministrativeExpenses', 'otherExpenses', 'operatingExpenses', 'costAndExpenses', 'netInterestIncome', 'interestIncome']

Balance Sheets:
  Date range: 1983-06-30 00:00:00 to 2026-01-25 00:00:00
  Unique tickers: 3088
  Columns: 62

Cash Flows:
  Date range: 2001-03-31 00:00:00 to 2026-01-24 00:00:00
  Unique tickers: 3082
  Columns: 48

AAPL EXAMPLE (Expected ~40 quarters)

AAPL quarters in income statement: 40

AAPL recent quarters:


,date,symbol,period,fiscalYear,revenue,netIncome
299,2025-12-27,AAPL,Q1,2026,1.437560e+11,4.209700e+10
300,2025-09-27,AAPL,Q4,2025,1.024660e+11,2.746600e+10
301,2025-06-28,AAPL,Q3,2025,9.403600e+10,2.343400e+10
302,2025-03-29,AAPL,Q2,2025,9.535900e+10,2.478000e+10
303,2024-12-28,AAPL,Q1,2025,1.243000e+11,3.633000e+10
304,2024-09-28,AAPL,Q4,2024,9.493000e+10,1.473600e+10
305,2024-06-29,AAPL,Q3,2024,8.577700e+10,2.144800e+10
306,2024-03-30,AAPL,Q2,2024,9.075300e+10,2.363600e+10
307,2023-12-30,AAPL,Q1,2024,1.195750e+11,3.391600e+10
308,2023-09-30,AAPL,Q4,2023,8.949800e+10,2.295600e+10



COLUMN VALIDATION
✅ Income has all required columns
✅ Balance has all required columns
✅ Cashflow has all required columns

✅ VERIFICATION COMPLETE
